<a href="https://colab.research.google.com/github/bihagkashikar/bits-pilani-mtech-genai-ml/blob/master/maths-assignment-01/q2_power_method.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q2: Rank, Covariance and the Power Method
This notebook contains the complete self-contained Python solution for Q2.

## Q2.1: Generate Dataset


In [ ]:
"""Q2.1: Generate X in R^(500 x 6) from four standard-normal features."""

import numpy as np


def generate_dataset(seed=41602):
    """Generate the four random features and the two dependent features."""
    rng = np.random.default_rng(seed)
    first_four = rng.standard_normal((500, 4))
    f1 = first_four[:, 0]
    f2 = first_four[:, 1]
    f3 = first_four[:, 2]
    f4 = first_four[:, 3]
    f5 = 2.0 * f1 + 3.0 * f2
    f6 = f3 - 2.0 * f4
    return np.column_stack((f1, f2, f3, f4, f5, f6))


data = generate_dataset()
np.set_printoptions(precision=8, suppress=False, linewidth=160)
print("Dataset shape:", data.shape)
print("First five rows of X:")
print(data[:5])

## Q2.2: Compute Rank


In [ ]:
"""Q2.2: Compute the rank of X using Gaussian elimination."""


def matrix_rank_by_elimination(matrix, tolerance=1e-10):
    """Compute matrix rank without using NumPy's matrix-rank helper."""
    work = matrix.astype(float).copy()
    row_count, column_count = work.shape
    pivot_row = 0
    rank = 0
    for column in range(column_count):
        if pivot_row == row_count:
            break
        candidate = pivot_row + int(np.argmax(np.abs(work[pivot_row:, column])))
        if abs(work[candidate, column]) <= tolerance:
            continue
        work[[pivot_row, candidate]] = work[[candidate, pivot_row]]
        for lower_row in range(pivot_row + 1, row_count):
            multiplier = work[lower_row, column] / work[pivot_row, column]
            work[lower_row, column:] -= multiplier * work[pivot_row, column:]
        rank += 1
        pivot_row += 1
    return rank


print("Rank of X:", matrix_rank_by_elimination(data))

## Q2.3(a): Covariance Matrix


In [ ]:
"""Q2.3(a): Compute C = (1/n) X^T X."""


def covariance_matrix(data):
    """Compute the covariance matrix specified in the question."""
    return (data.T @ data) / data.shape[0]


covariance = covariance_matrix(data)
print(covariance)

## Q2.3(b-c): Power Method and Deflation


In [ ]:
"""Q2.3(b-c): Approximate successive eigenpairs using power-method deflation."""


def power_method(matrix, tolerance=1e-7, max_iterations=100000):
    """Approximate the dominant eigenpair and return its iteration count."""
    vector = np.ones(matrix.shape[0], dtype=float)
    vector /= np.linalg.norm(vector)
    previous_eigenvalue = 0.0
    for iteration in range(1, max_iterations + 1):
        next_vector = matrix @ vector
        next_vector /= np.linalg.norm(next_vector)
        eigenvalue = float(next_vector @ matrix @ next_vector)
        if abs(eigenvalue - previous_eigenvalue) < tolerance:
            return eigenvalue, next_vector, iteration
        vector = next_vector
        previous_eigenvalue = eigenvalue
    raise RuntimeError("Power method did not converge within max_iterations.")


def successive_power_method(matrix, count, tolerance=1e-7):
    """Apply C - sum(v_j v_j^T C) to find successive eigenpairs."""
    eigenvalues = []
    eigenvectors = []
    deflated = matrix.copy()
    iteration_counts = []
    for _ in range(count):
        eigenvalue, eigenvector, iterations = power_method(deflated, tolerance)
        eigenvalues.append(eigenvalue)
        eigenvectors.append(eigenvector)
        iteration_counts.append(iterations)
        deflated = deflated - np.outer(eigenvector, eigenvector) @ deflated
    return np.array(eigenvalues), np.column_stack(eigenvectors), iteration_counts


power_values, power_vectors, iterations = successive_power_method(covariance, 6)
print("Power-method eigenvalues:", power_values)
print("Power-method eigenvectors (columns):")
print(power_vectors)
print("Iterations:", iterations)

## Q2.3(d): NumPy Comparison


In [ ]:
"""Q2.3(d): Compare the power-method results with NumPy's eigendecomposition."""

exact_values, exact_vectors = np.linalg.eigh(covariance)
order = np.argsort(exact_values)[::-1]
exact_values = exact_values[order]
exact_vectors = exact_vectors[:, order]
print("NumPy eigenvalues (descending):", exact_values)
print("Absolute eigenvalue differences:", np.abs(power_values - exact_values))
print("Eigenvector absolute dot products:")
for index in range(6):
    print(f"v{index + 1}:", f"{abs(power_vectors[:, index] @ exact_vectors[:, index]):.8f}")

## Q2.3(e): Iteration Comparison


In [ ]:
"""Display iterations required for the 1e-7 power-method tolerance."""
print("Iterations for accuracy 1.00000000e-07:", iterations)
